# Post 1 — Does an LLM-Wiki forget facts as it rewrites itself?

Runs a real LLM-Wiki rewrite loop (Hermes skill's update policy) on **SciFact-Open** and measures fact retention vs. number of rewrites.

**Two ways to run:**
- **API backend (default, no GPU):** Kilo Code free gateway (or any OpenAI-compatible provider) with **your own key**. Works on Kaggle CPU or a laptop.
- **Local backend (GPU):** 4-bit Qwen3 on a Kaggle T4.

Internet must be **ON** (Settings → Internet).

In [ ]:
# 1. Choose backend + install
BACKEND = 'api'   # 'api' (Kilo Code, no GPU)  |  'hf' (local Qwen3, needs GPU)
if BACKEND == 'api':
    !pip -q install requests scikit-learn matplotlib >/dev/null 2>&1
else:
    !pip -q install 'transformers>=4.51' accelerate bitsandbytes scikit-learn matplotlib >/dev/null 2>&1
    import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — switch accelerator to GPU')

In [ ]:
# 2. Get the experiment code (set REPO_URL to your published repo, or upload code/ as a Kaggle Dataset)
import os, sys, shutil
REPO_URL = ''  # e.g. 'https://github.com/<you>/llm-wiki-field-notes'
if REPO_URL:
    !rm -rf /kaggle/working/repo && git clone --depth 1 $REPO_URL /kaggle/working/repo
    CODE_DIR = '/kaggle/working/repo/linkedin/posts/post1-ai-memory-rot/code'
else:
    CODE_DIR = None
    for root, _, files in os.walk('/kaggle/input'):
        if 'wiki_decay.py' in files: CODE_DIR = root; break
    assert CODE_DIR, 'Set REPO_URL or upload code/ as a Kaggle dataset.'
shutil.copytree(CODE_DIR, '/kaggle/working/code', dirs_exist_ok=True)
sys.path.insert(0, '/kaggle/working/code'); os.chdir('/kaggle/working/code')
print('code ok; data exists:', os.path.exists('data/scifact_subset.json'))

In [ ]:
# 3. Config + key
import os
if BACKEND == 'api':
    # Kaggle: Add-ons -> Secrets -> add KILO_API_KEY, or paste below (never commit it!)
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['LLM_API_KEY'] = UserSecretsClient().get_secret('KILO_API_KEY')
    except Exception:
        os.environ['LLM_API_KEY'] = os.environ.get('LLM_API_KEY', 'PASTE_YOUR_KILO_KEY_HERE')
    os.environ['LLM_BASE_URL'] = 'https://api.kilo.ai/api/gateway'  # force (overrides any stale env)
    LARGE_MODEL = 'nvidia/nemotron-3-super-120b-a12b:free'   # fast, clean, capable free model
    SMALL_MODEL = 'poolside/laguna-xs.2:free'                # smaller free model = real size contrast
else:
    LARGE_MODEL = 'Qwen/Qwen3-8B'
    SMALL_MODEL = 'Qwen/Qwen3-1.7B'
ROUNDS, SEED, GROUP = 12, 0, 5   # first pass: try ROUNDS=8 to save time/calls
import wiki_decay as wd

In [ ]:
# 4. (API only) confirm the key works before the full run (shows which model was served)
if BACKEND == 'api':
    !LLM_BASE_URL=$LLM_BASE_URL LLM_API_KEY=$LLM_API_KEY python test_api.py --model "$LARGE_MODEL"

In [ ]:
# 5. Minimum-viable run: plain decay + rollback harness.
#    Note: nemotron-super is a 120B reasoning model (~11-20s/call) -> a 12-round run can take ~2-3h.
wd.run('plain',   LARGE_MODEL, ROUNDS, SEED, GROUP, backend=BACKEND)
wd.run('harness', LARGE_MODEL, ROUNDS, SEED, GROUP, backend=BACKEND)

In [ ]:
# 6. (Optional) small-model arm — 'bigger model won't save you'. Works on API too (real size contrast).
wd.run('plain', SMALL_MODEL, ROUNDS, SEED, GROUP, small=True, backend=BACKEND)

In [ ]:
# 7. Plot + headline numbers; persist outputs
!python plot.py
!cp -r ../results /kaggle/working/results 2>/dev/null; cp -r ../figures /kaggle/working/figures 2>/dev/null
from IPython.display import Image
Image('../figures/forgetting_curve.png')